# Layer 3 — Layered Feature Engineering dan Balancing

Notebook ini menyusun matriks pelatihan untuk XGBoost.

Satu baris adalah satu pasangan **pengguna × aturan asosiasi**. Label tidak bisa dibuat satu per pengguna, karena setiap aturan menunjuk ke produk consequent yang berbeda.

Fitur perilaku dihitung dari pesanan `prior`. Label `y` dihitung dari pesanan `train`:

- `y = 1` jika pengguna membeli produk consequent pada pesanan train;
- `y = 0` jika tidak.

Pengguna pada 50.000 pesanan prior pertama dikeluarkan. Keranjang mereka sudah dipakai untuk membentuk `apriori_rules.csv`, sehingga tidak ikut menjadi data pelatihan.

Dari pengguna train yang tersisa, diambil 12.000 pengguna secara acak. Satu pengguna menghasilkan 24 baris (satu per aturan). Batas ini menjaga SMOTE agar tidak memperbesar matriks hingga beberapa juta baris.

In [1]:
# ==========================================
# ENVIRONMENT CONFIGURATION
# Ubah menjadi True jika dijalankan di Server Kampus (RAM/CPU besar)
# Ubah menjadi False jika dijalankan di Laptop Lokal
# ==========================================
RUN_ON_SERVER = False

from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE

warnings.filterwarnings(
    "ignore",
    message="`BaseEstimator._validate_data` is deprecated",
    category=FutureWarning,
)

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. "
        "Buka Jupyter dari folder Cross Selling Retail."
    )

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "dataset"
RULES_PATH = PROJECT_DIR / "outputs" / "apriori_rules.csv"
OUTPUT_PATH = PROJECT_DIR / "outputs" / "layer3_smoted_features.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if not RULES_PATH.exists() or not (DATA_DIR / "orders.csv").exists():
    raise FileNotFoundError(
        "Jalankan notebook ini dari folder proyek Cross Selling Retail, "
        "setelah outputs/apriori_rules.csv tersedia."
    )

N_MINING_ORDERS = 50_000
MAX_USERS = 12_000
RANDOM_STATE = 42

if RUN_ON_SERVER:
    CHUNK_SIZE = 5_000_000
    print("--> [INFO] Berjalan dalam mode SERVER (Parameter Maksimal)")
else:
    CHUNK_SIZE = 1_000_000
    print("--> [INFO] Berjalan dalam mode LAPTOP (Parameter Terbatas)")

print(
    "--> [INFO] Parameter Layer 3 siap: 8 fitur, maksimal 12.000 pengguna, "
    f"CHUNK_SIZE={CHUNK_SIZE:,}, random_state=42."
)
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")

FEATURE_COLUMNS = [
    "total_prior_orders",
    "avg_basket_size",
    "avg_days_between",
    "antecedent_rate",
    "rule_confidence",
    "rule_lift",
    "interest_confidence",
    "interest_lift",
]

## 1. Aturan asosiasi dan katalog produk

Nama pada aturan disambungkan ke `product_id`. Hanya 11 produk yang muncul di 24 aturan, jadi pemindaian file prior nanti dibatasi pada produk tersebut.

In [2]:
print("--> [INFO] Memuat apriori_rules.csv dan memetakan nama produk ke product_id...")
rules = pd.read_csv(RULES_PATH)
if rules[["antecedents", "consequents"]].apply(lambda s: s.str.contains(r" \+ ")).any().any():
    raise ValueError("Notebook ini disusun untuk aturan dengan satu produk di tiap sisi.")

products = pd.read_csv(DATA_DIR / "products.csv", usecols=["product_id", "product_name"])
products["product_name"] = (
    products["product_name"].str.replace("\xa0", " ", regex=False).str.strip()
)
name_to_id = dict(zip(products["product_name"], products["product_id"].astype(np.int32)))

item_names = sorted(set(rules["antecedents"]) | set(rules["consequents"]))
missing_names = [name for name in item_names if name not in name_to_id]
if missing_names:
    raise ValueError(f"Nama produk pada aturan tidak ada di katalog: {missing_names}")

id_to_name = {int(name_to_id[name]): name for name in item_names}
relevant_ids = set(id_to_name)
print(rules.shape)
print(f"Produk pada aturan: {len(item_names)}")

(24, 10)
Produk pada aturan: 11


## 2. Populasi pengguna dan fitur perilaku dari orders

`total_prior_orders` adalah `max(order_number)` pada pesanan prior. `avg_days_between` adalah rata-rata `days_since_prior_order`. Pesanan pertama memang kosong pada kolom jeda, dan Pandas mengabaikan nilai kosong itu saat menghitung rata-rata.

In [3]:
print("--> [INFO] Memulai proses ekstraksi fitur perilaku pengguna dari orders.csv...")
orders = pd.read_csv(
    DATA_DIR / "orders.csv",
    usecols=["order_id", "user_id", "eval_set", "order_number", "days_since_prior_order"],
    dtype={
        "order_id": np.int32,
        "user_id": np.int32,
        "order_number": np.int16,
        "days_since_prior_order": np.float32,
    },
)
print(orders.shape)

mining_order_ids = set(
    orders.loc[orders["eval_set"].eq("prior"), "order_id"].head(N_MINING_ORDERS).tolist()
)
mining_users = set(orders.loc[orders["order_id"].isin(mining_order_ids), "user_id"].tolist())
train_users = set(orders.loc[orders["eval_set"].eq("train"), "user_id"].tolist())
eligible_users = np.array(sorted(train_users - mining_users), dtype=np.int32)
print(f"Pengguna train di luar sampel Apriori: {len(eligible_users):,}")

rng = np.random.default_rng(RANDOM_STATE)
if len(eligible_users) > MAX_USERS:
    sampled_users = np.sort(rng.choice(eligible_users, size=MAX_USERS, replace=False))
else:
    sampled_users = eligible_users
user_set = set(sampled_users.tolist())
print(f"Pengguna yang dipakai: {len(sampled_users):,}")

prior_orders = orders.loc[
    orders["eval_set"].eq("prior") & orders["user_id"].isin(user_set),
    ["order_id", "user_id", "order_number", "days_since_prior_order"],
]
print(prior_orders.shape)

user_features = prior_orders.groupby("user_id", sort=False).agg(
    total_prior_orders=("order_number", "max"),
    n_prior_orders=("order_id", "size"),
    avg_days_between=("days_since_prior_order", "mean"),
)
if (user_features["total_prior_orders"] != user_features["n_prior_orders"]).any():
    raise ValueError("max(order_number) tidak sama dengan jumlah pesanan prior.")
if user_features["avg_days_between"].isna().any():
    raise ValueError("Ada pengguna tanpa jeda hari pada pesanan prior.")
user_features = user_features.drop(columns="n_prior_orders")
print(user_features.shape)

del mining_order_ids, mining_users, train_users, eligible_users
gc.collect()

(3421083, 5)
Pengguna train di luar sampel Apriori: 129,071
Pengguna yang dipakai: 12,000
(187482, 4)
(12000, 2)


0

## 3. Ukuran keranjang dan frekuensi antecedent

File `order_products__prior.csv` dibaca per satu juta baris. Yang disimpan hanya pesanan pengguna terpilih. Untuk skor Apriori, yang disimpan lebih sempit lagi: baris yang `product_id`-nya muncul pada aturan.

In [4]:
print("--> [INFO] Menghitung rata-rata ukuran keranjang dan frekuensi pembelian antecedent...")
order_to_user = dict(
    zip(prior_orders["order_id"].to_numpy().tolist(), prior_orders["user_id"].to_numpy().tolist())
)
selected_order_ids = set(order_to_user)
basket_counts = {}
history_parts = []

for chunk in pd.read_csv(
    DATA_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": np.int32, "product_id": np.int32},
    chunksize=CHUNK_SIZE,
):
    sub = chunk.loc[chunk["order_id"].isin(selected_order_ids)]
    if sub.empty:
        continue
    for order_id, n_items in sub.groupby("order_id", sort=False).size().items():
        basket_counts[int(order_id)] = basket_counts.get(int(order_id), 0) + int(n_items)
    relevant = sub.loc[sub["product_id"].isin(relevant_ids), ["order_id", "product_id"]]
    if not relevant.empty:
        history_parts.append(relevant)

if len(basket_counts) != len(selected_order_ids):
    raise ValueError("Ada pesanan prior terpilih yang tidak punya item.")

basket_size = pd.Series(basket_counts, name="basket_size")
avg_basket = (
    basket_size.rename_axis("order_id")
    .reset_index()
    .assign(user_id=lambda frame: frame["order_id"].map(order_to_user))
    .groupby("user_id")["basket_size"]
    .mean()
)
user_features["avg_basket_size"] = avg_basket
print(user_features.shape)

product_events = pd.concat(history_parts, ignore_index=True)
product_events["user_id"] = product_events["order_id"].map(order_to_user).astype(np.int32)
product_events["product_name"] = product_events["product_id"].map(id_to_name)
product_history = (
    product_events.drop_duplicates(["order_id", "product_name"])
    .groupby(["user_id", "product_name"], sort=False)["order_id"]
    .nunique()
    .rename("n_orders")
    .reset_index()
)
print(product_history.shape)

del basket_counts, basket_size, history_parts, product_events, prior_orders, order_to_user
gc.collect()

(12000, 3)
(29966, 3)


0

## 4. Label target dari pesanan train

`antecedent_rate` adalah pangsa pesanan prior yang memuat produk antecedent.

Skor ketertarikan:

- `interest_confidence = antecedent_rate × confidence`
- `interest_lift = antecedent_rate × lift`

Keduanya nol bila pengguna tidak pernah membeli antecedent.

In [5]:
print("--> [INFO] Menyusun label target dari order_products__train.csv dan menggabungkan skor Apriori...")
train_orders = orders.loc[
    orders["eval_set"].eq("train") & orders["user_id"].isin(user_set),
    ["order_id", "user_id"],
]
print(train_orders.shape)

train_items = pd.read_csv(
    DATA_DIR / "order_products__train.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": np.int32, "product_id": np.int32},
)
train_items = train_items.loc[
    train_items["order_id"].isin(set(train_orders["order_id"].tolist()))
    & train_items["product_id"].isin(relevant_ids)
]
train_items = train_items.merge(train_orders, on="order_id", how="inner")
train_items["product_name"] = train_items["product_id"].map(id_to_name)
print(train_items.shape)

users = user_features.index.to_numpy()
n_users = len(users)
n_rules = len(rules)

purchase_counts = (
    product_history.pivot(index="user_id", columns="product_name", values="n_orders")
    .reindex(index=users, columns=item_names)
    .fillna(0)
    .to_numpy(dtype=np.float32)
)
train_flags = (
    train_items.assign(bought=np.int8(1))
    .pivot_table(
        index="user_id",
        columns="product_name",
        values="bought",
        aggfunc="max",
        fill_value=0,
    )
    .reindex(index=users, columns=item_names, fill_value=0)
    .to_numpy(dtype=np.int8)
)

name_to_col = {name: col for col, name in enumerate(item_names)}
antecedent_idx = np.array([name_to_col[name] for name in rules["antecedents"]], dtype=np.int16)
consequent_idx = np.array([name_to_col[name] for name in rules["consequents"]], dtype=np.int16)

total_orders = user_features.loc[users, "total_prior_orders"].to_numpy(dtype=np.float32)
avg_basket_size = user_features.loc[users, "avg_basket_size"].to_numpy(dtype=np.float32)
avg_days_between = user_features.loc[users, "avg_days_between"].to_numpy(dtype=np.float32)
antecedent_rate = purchase_counts[:, antecedent_idx] / total_orders[:, None]
rule_confidence = rules["confidence"].to_numpy(dtype=np.float32)
rule_lift = rules["lift"].to_numpy(dtype=np.float32)
target = train_flags[:, consequent_idx]

feature_df = pd.DataFrame(
    {
        "total_prior_orders": np.repeat(total_orders, n_rules),
        "avg_basket_size": np.repeat(avg_basket_size, n_rules),
        "avg_days_between": np.repeat(avg_days_between, n_rules),
        "antecedent_rate": antecedent_rate.reshape(-1),
        "rule_confidence": np.tile(rule_confidence, n_users),
        "rule_lift": np.tile(rule_lift, n_users),
        "interest_confidence": (antecedent_rate * rule_confidence).reshape(-1),
        "interest_lift": (antecedent_rate * rule_lift).reshape(-1),
        "y": target.reshape(-1),
    }
)
feature_df[FEATURE_COLUMNS] = feature_df[FEATURE_COLUMNS].astype(np.float32)
feature_df["y"] = feature_df["y"].astype(np.int8)

if feature_df.isna().any().any():
    raise ValueError("Matriks fitur masih berisi nilai kosong.")
if (feature_df["antecedent_rate"] < 0).any() or (feature_df["antecedent_rate"] > 1).any():
    raise ValueError("antecedent_rate berada di luar rentang 0 sampai 1.")

print(feature_df.shape)
print(feature_df["y"].value_counts().sort_index().to_string())
print(feature_df["y"].value_counts(normalize=True).sort_index().round(4).to_string())

del orders, train_orders, train_items, product_history, user_features
gc.collect()

(12000, 2)
(8949, 4)
(288000, 9)
y
0    259571
1     28429
y
0    0.9013
1    0.0987


0

## 5. SMOTE dan ekspor

Kelas 0 jauh lebih banyak daripada kelas 1. SMOTE membuat sampel sintetis kelas minoritas sampai jumlahnya sama dengan kelas mayoritas. Sampel sintetis bukan pengguna sungguhan, jadi `user_id` tidak ikut disimpan.

Hasilnya ditulis ke `outputs/layer3_smoted_features.csv`.

In [6]:
print("--> [INFO] Memeriksa proporsi kelas dan menerapkan SMOTE sampai kelas 0 dan kelas 1 seimbang...")
X = feature_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y = feature_df["y"].to_numpy(dtype=np.int8)
minority_n = int(np.bincount(y.astype(np.int32)).min())
if minority_n <= 1:
    raise ValueError("Kelas minoritas terlalu kecil untuk SMOTE.")

smote = SMOTE(
    sampling_strategy="auto",
    k_neighbors=min(5, minority_n - 1),
    random_state=RANDOM_STATE,
)
X_balanced, y_balanced = smote.fit_resample(X, y)

smoted_df = pd.DataFrame(X_balanced, columns=FEATURE_COLUMNS).astype(np.float32)
smoted_df["y"] = y_balanced.astype(np.int8)
smoted_df.to_csv(OUTPUT_PATH, index=False)

print(smoted_df.shape)
print(smoted_df["y"].value_counts().sort_index().to_string())
print(f"Tersimpan: {OUTPUT_PATH.name}")

(519142, 9)
y
0    259571
1    259571
Tersimpan: layer3_smoted_features.csv
